# Money Supply (Real M2) - Notebook 3: Backtest


## 1. Load Signals + Returns


In [ ]:
# Cell 1: Drive mount (top of every notebook)
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/macro_strategies/'
import os
os.makedirs(BASE_PATH + 'data/raw', exist_ok=True)
os.makedirs(BASE_PATH + 'data/processed', exist_ok=True)


In [ ]:
# Cell 2: Standard installs
!pip install -q fredapi vectorbt yfinance pandas-ta matplotlib seaborn pyarrow


In [ ]:
import pandas as pd
import numpy as np
import vectorbt as vbt
signal = pd.read_parquet(BASE_PATH + 'data/processed/signals.parquet')['signal']
returns = pd.read_parquet(BASE_PATH + 'data/processed/market_returns.parquet')


## 2. VectorBT Setup


In [ ]:
FEES = 0.0005       # 5 bps
SLIPPAGE = 0.001    # 10 bps round-trip
INIT_CASH = 100_000
FREQ = 'M'


## 3. Strategy Variants

(a) pure directional, (b) long-only overlay, (c) continuous sizing, (d) relative value (if multi-country).


In [ ]:
prices_cum  = (1 + returns['SPY']).cumprod() * 100
sig_aligned = signal.reindex(returns.index).ffill().dropna()
common      = prices_cum.index.intersection(sig_aligned.index)
prices_c    = prices_cum.loc[common]
sig_c       = sig_aligned.loc[common]

entries_dir = sig_c > 0
exits_dir   = sig_c < 0
entries_ov  = sig_c > 0.5
exits_ov    = sig_c < 0.0
position_size = sig_c.clip(-1, 1)

## 4. VectorBT Simulation


In [ ]:
pf_dir = vbt.Portfolio.from_signals(
    close=prices_c, entries=entries_dir, exits=exits_dir,
    freq=FREQ, init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE,
)
pf_ov = vbt.Portfolio.from_signals(
    close=prices_c, entries=entries_ov, exits=exits_ov,
    freq=FREQ, init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE,
)
print(pf_dir.stats())

## 5. Performance Metrics Table

Sharpe, Sortino, Max DD, Calmar, win rate, benchmark correlation.


In [ ]:
from src.reporting import performance_metrics, metrics_table_markdown

r_bench = returns['SPY'].loc[common]
for label, r in [('Directional', pf_dir.returns()),
                  ('Overlay',     pf_ov.returns()),
                  ('Benchmark (SPY)', r_bench)]:
    m = performance_metrics(r, freq=12, benchmark=r_bench)
    print(metrics_table_markdown(m, title=label))
    print()

## 6. Benchmark Comparison

PnL chart, drawdown chart, rolling 2Y Sharpe.


In [ ]:
from src.reporting import plot_cumulative_pnl, plot_drawdown
import matplotlib.pyplot as plt

returns_df = pd.DataFrame({
    'Directional': pf_dir.returns(),
    'Overlay':     pf_ov.returns(),
    'SPY':         r_bench,
}).dropna()
fig_pnl = plot_cumulative_pnl(returns_df, title='Cumulative PnL')
plt.show()
fig_dd = plot_drawdown(pf_dir.returns(), title='Directional Strategy Drawdown')
plt.show()

## 7. Robustness Checks

Walk-forward, parameter sensitivity, sub-period, tx-cost sensitivity.


In [ ]:
periods = [('Pre-GFC', '1995', '2007'), ('GFC', '2007', '2010'),
           ('Post-GFC', '2010', '2019'), ('COVID+Inflation', '2019', '2024')]
rows = []
for name, s, e in periods:
    r_sl = pf_dir.returns().loc[s:e]
    if len(r_sl) > 12:
        ann_r = (1 + r_sl.mean())**12 - 1
        ann_v = r_sl.std() * 12**0.5
        rows.append({'Period': name, 'Ann Ret': f'{ann_r:.1%}',
                     'Sharpe': f'{ann_r/ann_v:.2f}' if ann_v > 0 else 'n/a'})
print(pd.DataFrame(rows).to_string(index=False))

for fee_bps in [0, 5, 10, 20]:
    pf_tc = vbt.Portfolio.from_signals(
        close=prices_c, entries=entries_dir, exits=exits_dir,
        freq=FREQ, init_cash=INIT_CASH,
        fees=fee_bps/10000, slippage=SLIPPAGE,
    )
    r = pf_tc.returns()
    ann_r = (1 + r.mean())**12 - 1
    ann_v = r.std() * 12**0.5
    print(f'{fee_bps}bps fees: Sharpe = {ann_r/ann_v:.2f}' if ann_v > 0 else f'{fee_bps}bps: n/a')

## 8. Export


In [ ]:
import pickle

results = {'directional': pf_dir, 'overlay': pf_ov}
with open(BASE_PATH + 'data/processed/backtest_results.pkl', 'wb') as f:
    pickle.dump(results, f)
print('Backtest results saved.')

## Limitations

- Fees/slippage assumptions are indicative, not calibrated.
- Monthly frequency misses intra-month signal changes.
